## Dataset

Loads the enriched train/test CSVs saved at the end of notebook 04 (`enriched_train.csv` / `enriched_test.csv`), which already reflect the **one-month-forward split**: train = `CloseDate < 2026-06-01`, test = `CloseDate >= 2026-06-01`.

In [1]:
# Week 7 Advanced Models
# Trying Gradient Boosting (XGBoost) on top of the Week 6 enriched feature set
# Also trying LightGBM, to be in accordance with team

# Loads the enriched train/test datasets saved at the end of Week 6, rather than
# doing the entire preprocessing/feature-engineering again

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

train = pd.read_csv('/Users/jgd/IDXWORK/datasets/enriched/enriched_train.csv', low_memory=False)
test = pd.read_csv('/Users/jgd/IDXWORK/datasets/enriched/enriched_test.csv', low_memory=False)

print(train.shape)
print(test.shape)

(125795, 86)
(12408, 86)


In [3]:
# Same final feature set as the end of Week 6
numerical_features = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet',
                      'YearBuilt', 'Latitude', 'Longitude', 'AssociationFee', 'Stories',
                      'BedBathRatio', 'PropertyAge', 'BedroomAreaRatio', 'CloseMonthSin', 'CloseMonthCos']

boolean_features = ['FireplaceYN', 'NewConstructionYN', 'AttachedGarageYN', 'ViewYN', 'PoolPrivateYN']

categorical_features = ['CountyOrParish', 'DistrictGrouped']

target = 'ClosePrice'

features = numerical_features + boolean_features + categorical_features

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

In [4]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features + boolean_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(125795, 175)
(12408, 175)


In [5]:
# Baseline XGBoost - starting with conservative settings (bounded depth, modest number of trees)
# to avoid the same memory/runtime issue Random Forest caused in Week 6 with unbounded settings

from xgboost import XGBRegressor

xgb_baseline = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
xgb_baseline.fit(X_train_processed, y_train)

y_pred_xgb = xgb_baseline.predict(X_test_processed)

r2_xgb = r2_score(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)
mdape_xgb = np.median(np.abs((y_test - y_pred_xgb) / y_test))

print(f"XGBoost (baseline) R2: {r2_xgb:.4f}")
print(f"MAE: ${mae_xgb:,.0f}")
print(f"MAPE: {mape_xgb:.2%}")
print(f"MdAPE: {mdape_xgb:.2%}")

XGBoost (baseline) R2: 0.8455
MAE: $212,014
MAPE: 17.37%
MdAPE: 12.19%


In [6]:
# Light hyperparameter tuning - checking a small set of depth/learning_rate combinations via
# cross-validation, same approach as the Decision Tree depth sweep in Week 5

# Round 3: max_depth was still climbing at 10 (the top of round 2), so this adds max_depth=12 and 15
# to check whether it keeps improving, plateaus, or starts declining the way it did for the
# Decision Tree in Week 5 (which peaked at depth 15 and dipped by depth 20)


from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

depths = [3, 5, 7, 10, 12, 15]
learning_rates = [0.05, 0.1, 0.2]

for depth in depths:
    for lr in learning_rates:
        xgb_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', XGBRegressor(n_estimators=200, max_depth=depth, learning_rate=lr, random_state=42))
        ])
        cv_scores = cross_val_score(xgb_pipeline, X_train, y_train, cv=5, scoring='r2')
        print(f"max_depth={depth}, learning_rate={lr}: CV R2={cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

max_depth=3, learning_rate=0.05: CV R2=0.7612 (+/- 0.0058)


max_depth=3, learning_rate=0.1: CV R2=0.7980 (+/- 0.0064)


max_depth=3, learning_rate=0.2: CV R2=0.8270 (+/- 0.0068)


max_depth=5, learning_rate=0.05: CV R2=0.8246 (+/- 0.0072)


max_depth=5, learning_rate=0.1: CV R2=0.8483 (+/- 0.0065)


...

max_depth=12, learning_rate=0.1: CV R2=0.8796 (+/- 0.0050)


max_depth=12, learning_rate=0.2: CV R2=0.8749 (+/- 0.0069)


max_depth=15, learning_rate=0.05: CV R2=0.8719 (+/- 0.0055)


max_depth=15, learning_rate=0.1: CV R2=0.8728 (+/- 0.0051)


max_depth=15, learning_rate=0.2: CV R2=0.8680 (+/- 0.0063)


In [7]:
# tuned model - max_depth=10, learning_rate=0.1 scored best across all 18 combinations tested
# (3 rounds of sweeping, depths 3 through 15). Deeper trees (12, 15) were checked and scored lower,
# confirming depth=10 is a genuine peak rather than an untested edge of the range.

xgb_final = XGBRegressor(n_estimators=200, max_depth=10, learning_rate=0.1, random_state=42)
xgb_final.fit(X_train_processed, y_train)

y_pred_xgb_final = xgb_final.predict(X_test_processed)

r2_xgb_final = r2_score(y_test, y_pred_xgb_final)
mae_xgb_final = mean_absolute_error(y_test, y_pred_xgb_final)
mape_xgb_final = mean_absolute_percentage_error(y_test, y_pred_xgb_final)
mdape_xgb_final = np.median(np.abs((y_test - y_pred_xgb_final) / y_test))

print(f"XGBoost (tuned) R2: {r2_xgb_final:.4f}")
print(f"MAE: ${mae_xgb_final:,.0f}")
print(f"MAPE: {mape_xgb_final:.2%}")
print(f"MdAPE: {mdape_xgb_final:.2%}")

XGBoost (tuned) R2: 0.8851
MAE: $167,793
MAPE: 12.70%
MdAPE: 8.48%


In [8]:
# Baseline LightGBM with same conservative settings as the XGBoost baseline for a fair comparison

from lightgbm import LGBMRegressor

lgbm_baseline = LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_baseline.fit(X_train_processed, y_train)

y_pred_lgbm = lgbm_baseline.predict(X_test_processed)

r2_lgbm = r2_score(y_test, y_pred_lgbm)
mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)
mape_lgbm = mean_absolute_percentage_error(y_test, y_pred_lgbm)
mdape_lgbm = np.median(np.abs((y_test - y_pred_lgbm) / y_test))

print(f"LightGBM (baseline) R2: {r2_lgbm:.4f}")
print(f"MAE: ${mae_lgbm:,.0f}")
print(f"MAPE: {mape_lgbm:.2%}")
print(f"MdAPE: {mdape_lgbm:.2%}")

LightGBM (baseline) R2: 0.8493
MAE: $210,444
MAPE: 17.23%
MdAPE: 12.03%


In [9]:
# Same depth/learning_rate sweep as the XGBoost tuning above
depths = [3, 5, 7, 10, 12, 15]
learning_rates = [0.05, 0.1, 0.2]

for depth in depths:
    for lr in learning_rates:
        lgbm_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', LGBMRegressor(n_estimators=200, max_depth=depth, learning_rate=lr, random_state=42, verbose=-1))
        ])
        cv_scores = cross_val_score(lgbm_pipeline, X_train, y_train, cv=5, scoring='r2')
        print(f"max_depth={depth}, learning_rate={lr}: CV R2={cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

max_depth=3, learning_rate=0.05: CV R2=0.7616 (+/- 0.0065)


max_depth=3, learning_rate=0.1: CV R2=0.7990 (+/- 0.0072)


max_depth=3, learning_rate=0.2: CV R2=0.8258 (+/- 0.0049)


max_depth=5, learning_rate=0.05: CV R2=0.8247 (+/- 0.0061)


max_depth=5, learning_rate=0.1: CV R2=0.8473 (+/- 0.0063)


...

max_depth=12, learning_rate=0.1: CV R2=0.8714 (+/- 0.0049)


max_depth=12, learning_rate=0.2: CV R2=0.8775 (+/- 0.0057)


max_depth=15, learning_rate=0.05: CV R2=0.8560 (+/- 0.0059)


max_depth=15, learning_rate=0.1: CV R2=0.8716 (+/- 0.0054)


max_depth=15, learning_rate=0.2: CV R2=0.8776 (+/- 0.0059)


In [10]:
# tuned model - max_depth=15, learning_rate=0.2 scored best across all 18 combinations tested.
# Unlike XGBoost, R2 was still climbing at depth=15 (the top of the tested range) rather than
# plateauing, so this may not be a genuine peak - a wider sweep could still find something better.

lgbm_final = LGBMRegressor(n_estimators=200, max_depth=15, learning_rate=0.2, random_state=42, verbose=-1)
lgbm_final.fit(X_train_processed, y_train)

y_pred_lgbm_final = lgbm_final.predict(X_test_processed)

r2_lgbm_final = r2_score(y_test, y_pred_lgbm_final)
mae_lgbm_final = mean_absolute_error(y_test, y_pred_lgbm_final)
mape_lgbm_final = mean_absolute_percentage_error(y_test, y_pred_lgbm_final)
mdape_lgbm_final = np.median(np.abs((y_test - y_pred_lgbm_final) / y_test))

print(f"LightGBM (tuned) R2: {r2_lgbm_final:.4f}")
print(f"MAE: ${mae_lgbm_final:,.0f}")
print(f"MAPE: {mape_lgbm_final:.2%}")
print(f"MdAPE: {mdape_lgbm_final:.2%}")

LightGBM (tuned) R2: 0.8868
MAE: $180,152
MAPE: 14.49%
MdAPE: 10.32%


In [11]:
# Comparison against the best Week 6 model (Random Forest, 100 trees, depth 15)
# Week 6 numbers copied from the Weeks5and6/04_model_comparison.ipynb results

week7_comparison = pd.DataFrame({
    'Model': ['Random Forest (Week 6)', 'XGBoost (baseline)', 'XGBoost (tuned)', 'LightGBM (baseline)', 'LightGBM (tuned)'],
    'R2': [0.856809, r2_xgb, r2_xgb_final, r2_lgbm, r2_lgbm_final],
    'MAE': [197695.089079, mae_xgb, mae_xgb_final, mae_lgbm, mae_lgbm_final],
    'MAPE': [0.157561, mape_xgb, mape_xgb_final, mape_lgbm, mape_lgbm_final],
    'MdAPE': [0.102295, mdape_xgb, mdape_xgb_final, mdape_lgbm, mdape_lgbm_final],
})

week7_comparison['R2'] = week7_comparison['R2'].map(lambda x: f"{x:.4f}")
week7_comparison['MAE'] = week7_comparison['MAE'].map(lambda x: f"${x:,.0f}")
week7_comparison['MAPE'] = week7_comparison['MAPE'].map(lambda x: f"{x:.2%}")
week7_comparison['MdAPE'] = week7_comparison['MdAPE'].map(lambda x: f"{x:.2%}")

week7_comparison

,Model,R2,MAE,MAPE,MdAPE
0,Random Forest (Week 6),0.8568,"$197,695",15.76%,10.23%
1,XGBoost (baseline),0.8455,"$212,014",17.37%,12.19%
2,XGBoost (tuned),0.8851,"$167,793",12.70%,8.48%
3,LightGBM (baseline),0.8493,"$210,444",17.23%,12.03%
4,LightGBM (tuned),0.8868,"$180,152",14.49%,10.32%


## Week 7 - Advanced Models: XGBoost & LightGBM Behavior

| Model | R2 | MAE | MAPE | MdAPE |
|---|---|---|---|---|
| Random Forest (Week 6) | 0.8568 | `$197,695` | 15.76% | 10.23% |
| XGBoost (baseline, untuned) | 0.8455 | `$212,014` | 17.37% | 12.19% |
| XGBoost (tuned, final) | 0.8851 | `$167,793` | 12.70% | 8.48% |
| LightGBM (baseline, untuned) | 0.8493 | `$210,444` | 17.23% | 12.03% |
| LightGBM (tuned, final) | 0.8868 | `$180,152` | 14.49% | 10.32% |

Both gradient boosting models beat Random Forest once tuned. XGBoost (tuned) has the best MAE/MAPE/MdAPE of everything tried, while LightGBM (tuned) edges it out slightly on R2 (0.8868 vs 0.8851) but with higher dollar-error metrics - the two aren't strictly ordered, they're trading off differently across metrics.

Note on LightGBM tuning: its sweep didn't plateau the way XGBoost's did. XGBoost's CV R2 clearly peaked at depth=10 and declined at 12/15. LightGBM's CV R2 was still climbing at depth=15 (the deepest tested), so `max_depth=15, learning_rate=0.2` is the best of what was tried, not a confirmed peak - a wider sweep (deeper trees and/or higher learning rates) could still find something better.